# Ropedia Academy — A6 · Rotations & motion representations

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChaoYue0307/ropedia-academy/blob/main/notebooks/A6.ipynb)

> **Implements the continuous 6D→matrix fix, verifies the result is a valid rotation, and draws the decoded orthonormal frame in 3D.**
>
> 实现连续的 6D→矩阵修复，验证结果是有效旋转，并在 3D 中画出解码得到的正交标架。

This is the lesson's core example — **self-contained and runnable end to end**. It builds toy tensors, performs the lesson's key computation, and **visualizes the result with matplotlib** (the plot renders inline below the cell), so you learn the concept by executing and *seeing* it.

Colab's default runtime already includes `torch`, `numpy`, `networkx`, and `matplotlib`, so just press **Run all** — every cell goes green and a figure appears. Sizes are shrunk to run on CPU; swap in a real batch and the same code scales up.

🔗 Full lesson (with the interactive demo & key terms): https://chaoyue0307.github.io/ropedia-academy/lesson/A6

In [ ]:
import torch, torch.nn.functional as F, matplotlib.pyplot as plt

# How you ENCODE rotation decides if a net can learn it. Euler -> gimbal lock;
# quaternion -> double cover; ANY <=4D encoding is discontinuous.
# Fix: regress a continuous 6D vector, Gram-Schmidt -> a valid rotation matrix.
def sixd_to_R(d6):
    a1, a2 = d6[..., :3], d6[..., 3:]
    b1 = F.normalize(a1, dim=-1)
    b2 = F.normalize(a2 - (b1*a2).sum(-1, keepdim=True)*b1, dim=-1)
    return torch.stack([b1, b2, torch.cross(b1, b2, dim=-1)], dim=-1)

R = sixd_to_R(torch.randn(6))
print("orthonormal:", torch.allclose(R @ R.T, torch.eye(3), atol=1e-5),
      "| det=+1:", bool(torch.allclose(R.det(), torch.tensor(1.), atol=1e-5)))

# Visualize: the 6D vector decodes to a valid orthonormal frame (rotated x/y/z axes).
fig = plt.figure(figsize=(4.5, 4.5)); ax = fig.add_subplot(projection='3d')
for i, c in enumerate(['r','g','b']):
    ax.quiver(0,0,0, *R[:,i], color=c, label=f"axis {['x','y','z'][i]}")
ax.set_xlim(-1,1); ax.set_ylim(-1,1); ax.set_zlim(-1,1); ax.legend()
plt.title("continuous 6D -> valid orthonormal rotation frame"); plt.show()

### Where to go next

- Swap the toy tensors for a real batch and watch the shapes flow through.
- Open the matching lesson for the math and an interactive figure: https://chaoyue0307.github.io/ropedia-academy/lesson/A6
- Browse every notebook: https://github.com/ChaoYue0307/ropedia-academy/tree/main/notebooks